# 04b — Phase 4A: which session statistics give the best live angle?

All three models are I_ML without look-ahead (200 trees × 31 leaves), **trained and tested
on streaming features** of the same variant, on people they never saw, on the same frames.

| name | bone lengths and scale come from | length unit |
|---|---|---|
| `window` (Phase 2) | the last 300 frames (~10 s); older frames are forgotten | shoulder width |
| `exp_sw` | **every frame since the session started** | shoulder width |
| `exp_torso` | every frame since the session started | **torso length** (mid-shoulder to mid-hip, as in `pose_normalise.py`); shoulder width until the hips have been seen |

No pose is asked of the patient: the session estimates simply keep improving as they exercise.

**Rule fixed in advance:** keep the variant with the lowest side-on (profile) error in E3;
if `exp_torso` is within 0.2° of the best, keep `exp_torso`, because it never forgets and
needs no pose from the patient.

In [2]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
import lightgbm as lgb

REPO  = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-'   # <-- your repo
FILES = {'rehab24': os.path.join(REPO, 'data', 'fit3d', 'feat_rehab24_stream.npz'),
         'fit3d':   os.path.join(REPO, 'data', 'fit3d', 'feat_fit3d_stream.npz')}
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
sys.path[:0] = [REPO, os.path.join(REPO, 'scripts')]
from calibration import fit_curve, apply_calibration
from fit_calibration import extend_below
pd.set_option('display.width', 220)

def view_class(v):
    """camera-relative view (REHAB24-6 cam18 labels describe the subject's pose to cam17)"""
    v = str(v).lower(); cam18 = 'cam18' in v
    if 'half' in v: return 'half'
    if 'front' in v: return 'profile' if cam18 else 'front'
    if 'prof' in v or 'side' in v: return 'front' if cam18 else 'profile'
    return 'unknown'


V1 = {'rehab24': 'feat_rehab24_stream.npz', 'fit3d': 'feat_fit3d_stream.npz'}       # Phase 2
V2 = {'rehab24': 'feat_rehab24_stream_v2.npz', 'fit3d': 'feat_fit3d_stream_v2.npz'} # Phase 4A
VARS = {'window': 'X_stream', 'exp_sw': 'X_exp_sw', 'exp_torso': 'X_exp_torso'}
frames = []
for ds in V1:
    p1, p2 = (os.path.join(REPO, 'data', 'fit3d', v[ds]) for v in (V1, V2))
    if not (os.path.exists(p1) and os.path.exists(p2)):
        print('missing (skipped):', p1 if not os.path.exists(p1) else p2); continue
    d1, d2 = np.load(p1, allow_pickle=False), np.load(p2, allow_pickle=False)
    names = [str(n) for n in d1['feature_names']]
    # match rows by (recording, frame number inside the recording), not by position:
    # the recordings can come out of Drive in a different order in another Colab session
    k1 = pd.DataFrame({'recording': d1['recording']}); k1['i'] = k1.groupby('recording').cumcount()
    k2 = pd.DataFrame({'recording': d2['recording']}); k2['i'] = k2.groupby('recording').cumcount()
    k2['pos'] = np.arange(len(k2))
    m = k1.merge(k2, on=['recording', 'i'], how='left')
    have = m['pos'].notna().to_numpy()
    pos = m['pos'].fillna(0).astype(int).to_numpy()
    y2 = np.where(have, d2['y'][pos], np.nan)
    same_y = have & np.isclose(d1['y'], y2, atol=1e-3)
    bad_rec = sorted(set(d1['recording'][have & ~same_y]))       # same name, different truth
    keep = same_y & ~np.isin(d1['recording'], bad_rec)
    n1, n2 = set(d1['recording']), set(d2['recording'])
    print(f"{ds}: v1 {len(d1['y']):,} rows / {len(n1)} recordings, v2 {len(d2['y']):,} rows / {len(n2)} recordings")
    print(f"   order identical: {len(d1['y']) == len(d2['y']) and (d1['recording'] == d2['recording']).all()}")
    print(f"   only in v1: {len(n1 - n2)}  only in v2: {len(n2 - n1)}  truth differs: {len(bad_rec)}")
    print(f"   rows used: {keep.sum():,} of {len(keep):,}")
    if keep.mean() < 0.9:
        print('   WARNING: under 90% of rows match - check that STEP 1-5 settings were the same in both Colab runs')
    df = pd.concat([pd.DataFrame(d1['X_stream'][keep], columns=names).add_prefix('window_'),
                    pd.DataFrame(d2['X_exp_sw'][pos[keep]], columns=names).add_prefix('exp_sw_'),
                    pd.DataFrame(d2['X_exp_torso'][pos[keep]], columns=names).add_prefix('exp_torso_')], axis=1)
    for k in ('y', 'person', 'recording', 'view', 'exercise', 'dataset'):
        df[k] = d1[k][keep]
    frames.append(df)
D = pd.concat(frames, ignore_index=True)
D['vclass'] = D['view'].map(view_class)
D = D[np.isfinite(D['window_angle_2d']) & np.isfinite(D['y'])].reset_index(drop=True)
DROP = {'bone_upper', 'bone_fore', 'sw_frac', 'is_right', 'std_angle', 'd_angle'}
FEATS = [n for n in names if n not in DROP]
print(D.groupby(['dataset', 'vclass']).agg(frames=('y', 'size'), people=('person', 'nunique')))

rehab24: v1 149,930 rows / 56 recordings, v2 149,930 rows / 56 recordings
   order identical: False
   only in v1: 0  only in v2: 0  truth differs: 0
   rows used: 149,930 of 149,930
fit3d: v1 58,345 rows / 129 recordings, v2 58,345 rows / 129 recordings
   order identical: True
   only in v1: 0  only in v2: 0  truth differs: 0
   rows used: 58,345 of 58,345
                 frames  people
dataset vclass                 
fit3d   front     54867       8
        half       3401       4
rehab24 front     85276      19
        half      23287       3
        profile   41265      16


## How stable are the length features over a session?

In [3]:
fps = np.where(D['dataset'] == 'fit3d', 25.0, 30.0)
D['t_s'] = D.groupby('recording').cumcount() / fps
band = pd.cut(D['t_s'], [0, 1, 3, 10, 1e9], labels=['0-1 s', '1-3 s', '3-10 s', '>10 s'], right=False)
# spread of the per-recording bone-length estimate: a good estimate settles and stays put
for v in VARS:
    cv = (D.groupby(['recording', band], observed=True)[v + '_plan_fore']
            .median().groupby(level=1, observed=True).std())
    print(f'{v:10s} spread of median forearm planarity by time:', cv.round(3).to_dict())

window     spread of median forearm planarity by time: {'0-1 s': 0.03999999910593033, '1-3 s': 0.11500000208616257, '3-10 s': 0.1080000028014183, '>10 s': 0.06300000101327896}
exp_sw     spread of median forearm planarity by time: {'0-1 s': 0.03999999910593033, '1-3 s': 0.11500000208616257, '3-10 s': 0.1080000028014183, '>10 s': 0.0689999982714653}
exp_torso  spread of median forearm planarity by time: {'0-1 s': 0.03999999910593033, '1-3 s': 0.11500000208616257, '3-10 s': 0.1080000028014183, '>10 s': 0.0689999982714653}


## Train and test every variant

In [4]:
HP = dict(objective='l1', n_estimators=200, num_leaves=31, learning_rate=0.09, min_child_samples=80,
          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbosity=-1, n_jobs=N_JOBS)

def fit(tr, v):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return lgb.LGBMRegressor(**HP).fit(tr[[f'{v}_{f}' for f in FEATS]].to_numpy(), tr['y'])

def curve_pred(tr, te):
    def one(a, b):
        for mc in (300, 100, 30):
            try:
                return extend_below(fit_curve(a, b, min_count=mc))
            except ValueError:
                pass
    pooled = one(tr['window_angle_2d'], tr['y']); out = np.empty(len(te))
    for vc, idx in te.groupby('vclass').indices.items():
        sub = tr[tr['vclass'] == vc]
        p = one(sub['window_angle_2d'], sub['y']) if len(sub) >= 2000 else None
        out[idx] = apply_calibration(te['window_angle_2d'].to_numpy(float)[idx], p or pooled)
    return out

def run(train_mask, test_mask, label):
    tr, te = D[train_mask], D[test_mask].reset_index(drop=True)
    out = te[['dataset', 'person', 'recording', 'vclass', 'y']].copy()
    out['curve'] = curve_pred(tr, te)
    for v in VARS:
        out[v] = np.clip(fit(tr, v).predict(te[[f'{v}_{f}' for f in FEATS]].to_numpy()), 0, 180)
    out['experiment'] = label
    return out

t0 = time.time()
is_fit, is_reh = D['dataset'] == 'fit3d', D['dataset'] == 'rehab24'
parts = []
if is_fit.any() and is_reh.any():
    parts += [run(is_reh, is_fit, 'E2a REHAB->Fit3D'), run(is_fit, is_reh, 'E2b Fit3D->REHAB')]
for p in sorted(D['person'].unique()):
    parts.append(run(D['person'] != p, D['person'] == p, 'E3 LOSO'))
    print('  held out', p, f'{time.time() - t0:.0f}s', flush=True)
P = pd.concat(parts, ignore_index=True)

  held out fit3d_s03 14s
  held out fit3d_s04 21s
  held out fit3d_s05 27s
  held out fit3d_s07 42s
  held out fit3d_s08 60s
  held out fit3d_s09 78s
  held out fit3d_s10 92s
  held out fit3d_s11 107s
  held out rehab_PM_003 120s
  held out rehab_PM_004 127s
  held out rehab_PM_010 134s
  held out rehab_PM_011 141s
  held out rehab_PM_013 158s
  held out rehab_PM_014 179s
  held out rehab_PM_025 196s
  held out rehab_PM_026 206s
  held out rehab_PM_030 213s
  held out rehab_PM_031 220s
  held out rehab_PM_033 237s
  held out rehab_PM_040 254s
  held out rehab_PM_044 272s
  held out rehab_PM_045 291s
  held out rehab_PM_102 308s
  held out rehab_PM_107 315s
  held out rehab_PM_108 324s
  held out rehab_PM_110 333s
  held out rehab_PM_115 344s
  held out rehab_PM_119 355s
  held out rehab_PM_121 366s
  held out rehab_PM_123 376s


## Results

In [5]:
M = ['curve'] + list(VARS)
err = P[M].sub(P['y'], axis=0).abs()
rec = pd.concat([P[['experiment', 'dataset', 'person', 'recording', 'vclass']], err], axis=1) \
        .groupby(['experiment', 'dataset', 'person', 'recording', 'vclass'], as_index=False)[M].mean()
tab = lambda r: r.groupby('experiment')[M].median().round(2).assign(recordings=r.groupby('experiment').size())
print('ALL VIEWS - median per-recording MAE (deg)'); display(tab(rec))
print('PROFILE VIEW (live setup)'); display(tab(rec[rec.vclass == 'profile']))

e3 = rec[rec.experiment == 'E3 LOSO']
rows = []
for scope, r in (('all views', e3), ('profile', e3[e3.vclass == 'profile'])):
    s = r.groupby('person')[M].mean()
    for a, b in (('window', 'exp_sw'), ('window', 'exp_torso'), ('curve', 'exp_torso')):
        g = s[a] - s[b]
        rows.append(dict(scope=scope, compare=f'{a} -> {b}', people=len(s), median_gain=round(float(g.median()), 2),
                         better_for=f'{int((g > 0).sum())}/{len(s)}', wilcoxon_p=round(float(wilcoxon(g).pvalue), 4)))
display(pd.DataFrame(rows))

ALL VIEWS - median per-recording MAE (deg)


,curve,window,exp_sw,exp_torso,recordings
experiment,,,,,
E2a REHAB->Fit3D,12.31,12.64,12.19,12.44,129
E2b Fit3D->REHAB,15.00,14.35,14.36,13.78,56
E3 LOSO,12.42,9.49,9.28,9.54,185


PROFILE VIEW (live setup)


,curve,window,exp_sw,exp_torso,recordings
experiment,,,,,
E2b Fit3D->REHAB,15.96,13.79,14.00,13.26,16
E3 LOSO,15.79,11.83,11.59,11.46,16


,scope,compare,people,median_gain,better_for,wilcoxon_p
0,all views,window -> exp_sw,30,0.18,22/30,0.2054
1,all views,window -> exp_torso,30,-0.00,15/30,0.7303
2,all views,curve -> exp_torso,30,3.73,29/30,0.0000
3,profile,window -> exp_sw,16,0.01,9/16,0.9799
4,profile,window -> exp_torso,16,0.70,10/16,0.6685
5,profile,curve -> exp_torso,16,4.36,13/16,0.0507


In [6]:
prof = e3[e3.vclass == 'profile'][list(VARS)].median()
best = prof.idxmin()
CHOICE = 'exp_torso' if prof['exp_torso'] - prof[best] <= 0.2 else best
print('E3 profile median error:', prof.round(2).to_dict())
print('CHOICE:', CHOICE)

E3 profile median error: {'window': 11.83, 'exp_sw': 11.59, 'exp_torso': 11.46}
CHOICE: exp_torso


## Retrain the deployable model with the chosen variant (REHAB24-6 only)

In [7]:
reh = D[D['dataset'] == 'rehab24']
model = fit(reh, CHOICE)
mode = {'window': {'session': 'window', 'scale': 'shoulder'},
        'exp_sw': {'session': 'expanding', 'scale': 'shoulder'},
        'exp_torso': {'session': 'expanding', 'scale': 'torso'}}[CHOICE]
path = os.path.join(REPO, 'data', f'iml_nolook_rehab24_{CHOICE}.txt')
model.booster_.save_model(path)
json.dump({'features': FEATS, 'feature_stream': mode,          # IMLStream rebuilds the same features
           'params': {k: v for k, v in HP.items() if k != 'n_jobs'},
           'trained_on': 'REHAB24-6 only (CC BY-NC), streaming features',
           'cv': {'E3_all_views_deg': round(float(e3[CHOICE].median()), 2),
                  'E3_profile_deg': round(float(prof[CHOICE]), 2)}},
          open(path + '.json', 'w'), indent=1)
print('saved', path)
print(f'use it live:  python demo_cv.py --variant ML --iml-model data/iml_nolook_rehab24_{CHOICE}.txt')

saved F:\namal sir project\github repo\AI-based-Physiotherapy-project-\data\iml_nolook_rehab24_exp_torso.txt
use it live:  python demo_cv.py --variant ML --iml-model data/iml_nolook_rehab24_exp_torso.txt


In [8]:
# optional: MLflow
try:
    import mlflow
    mlflow.set_tracking_uri('sqlite:///' + os.path.join(REPO, 'mlflow.db').replace('\\', '/'))
    mlflow.set_experiment('Physio-ROM-Estimation')
    with mlflow.start_run(run_name='phase4a-expanding-session-stats'):
        mlflow.set_tags({'stage': 'offline_evaluation', 'phase': '4A', 'choice': CHOICE})
        for scope, r in (('all', rec), ('profile', rec[rec.vclass == 'profile'])):
            for exp, row in tab(r).iterrows():
                for col in M:
                    mlflow.log_metric(f'{scope}_{exp.split()[0]}_{col}', float(row[col]))
        mlflow.log_artifact(path, artifact_path='model'); mlflow.log_artifact(path + '.json', artifact_path='model')
    print('logged to MLflow')
except ImportError:
    print('mlflow not installed - skipped')

logged to MLflow
